# Esta parte do código se refere à pipeline da camada GOLD em BATCH para testes antes de subir ao AWS

In [1]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Instalando as dependências
# ~~~~~~~~~~~~~~~~~~~~~~~~~~

# pyarrow para salvar em PARQUET

!pip install pyarrow colorama tabulate --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\carol\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# Importações
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
import logging
import time
import os
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

In [3]:
# ~~~~~~~~~~~~~~~
# CONFIGURAÇÕES
# ~~~~~~~~~~~~~~~
from pathlib import Path
from datetime import datetime

DATA_SILVER = Path("silver")
DATA_GOLD = Path("gold")

DATA_GOLD.mkdir(parents=True, exist_ok=True)

PROCESSAMENTO = datetime.now()

In [4]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
# CONFIGURAÇÃO DOS LOGS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s"
)

log = logging.getLogger(__name__)

In [5]:
# ~~~~~~~~~~~~~~~
# LOG INICIAL
# ~~~~~~~~~~~~~~~

log.info("~" * 35)
log.info("INICIANDO ETL DA CAMADA GOLD")
log.info("~" * 35)

2026-09-06 21:54:58,124 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-09-06 21:54:58,125 | INFO     | INICIANDO ETL DA CAMADA GOLD
2026-09-06 21:54:58,126 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


In [6]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# OBSERVABILIDADE: MÉTRICAS ESTRUTURADAS E ALERTAS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# (mesmo padrão usado nas camadas Bronze e Silver)

def log_metrica(evento, **campos):
    """
    Loga um evento estruturado (campo=valor), permitindo consulta via
    CloudWatch Logs Insights, ex.:
        fields @timestamp, tabela, volume, latencia_segundos
        | filter evento = "tabela_processada"
    """
    campos_formatados = " | ".join(f"{chave}={valor}" for chave, valor in campos.items())
    log.info(f"[METRICA] evento={evento} | {campos_formatados}")


def emitir_alerta(mensagem, **contexto):
    """
    Emite um alerta de erro (sempre em nível ERROR no log) e tenta
    publicar em um tópico SNS, se configurado via variável de ambiente
    SNS_TOPIC_ARN, para que a falha não dependa de alguém checar o log
    manualmente.
    """
    contexto_formatado = " | ".join(f"{k}={v}" for k, v in contexto.items())
    log.error(f"[ALERTA] {mensagem} | {contexto_formatado}")

    topico_sns = os.environ.get("SNS_TOPIC_ARN")

    if not topico_sns:
        log.warning("[ALERTA] SNS_TOPIC_ARN não configurado - alerta ficou registrado apenas no log")
        return

    try:
        import boto3
        sns = boto3.client("sns")
        sns.publish(
            TopicArn=topico_sns,
            Subject="[Tech Challenge] Falha no pipeline",
            Message=f"{mensagem}\n\n{contexto_formatado}"
        )
        log.info("[ALERTA] Notificação SNS publicada com sucesso")
    except Exception as e:
        log.warning(f"[ALERTA] Falha ao publicar no SNS (alerta permanece apenas no log): {e}")

In [7]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# LENDO ARQUIVOS DA CAMADA SILVER
"""
    Lê um arquivo Parquet da camada SILVER.

    Args:
        tabela (str): Nome da tabela.
    """
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def ler_silver(tabela):

    caminho = DATA_SILVER / f"{tabela}.parquet"

    log.info(f"Lendo Silver: {caminho}")

    return pd.read_parquet(caminho)

In [8]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - RANKING UF
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_ranking_uf():

    log.info("Construindo Gold: Ranking UF")

    df = ler_silver("uf")

    # removendo o Total, pois só há uma única informação
    df = df[
        df["rede"] != "Total (Federal, Estadual, Municipal e Privada)"
    ].copy()

    # ordena
    df = df.sort_values(
        by=["ano", "rede", "taxa_alfabetizacao"],
        ascending=[True, True, False]
    )

    # ranking por ano e rede
    df["ranking"] = (
        df.groupby(
            ["ano", "rede"]
        )["taxa_alfabetizacao"]
        .rank(
            method="dense",
            ascending=False
        )
        .astype(int)
    )

    df["_gold_processed_at"] = datetime.now()
    
    df = df[
        [
            "ano",
            "sigla_uf",
            "sigla_uf_nome",
            "rede",
            "taxa_alfabetizacao",
            "ranking",
            "_gold_processed_at"
        ]
    ]

    log.info("Ranking UF criado")

    return df

In [9]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - RANKING MUNICÍPIOS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_ranking_municipio():

    log.info("Construindo Gold: Ranking Municípios")

    df = ler_silver("municipio")

    # Ordena por ano, rede e taxa
    df = df.sort_values(
        by=["ano", "rede", "taxa_alfabetizacao"],
        ascending=[True, True, False]
    )

    # Cria ranking por ano e rede
    df["ranking"] = (
        df.groupby(
            ["ano", "rede"]
        )["taxa_alfabetizacao"]
        .rank(
            method="dense",
            ascending=False
        )
        .astype(int)
    )

    df["_gold_processed_at"] = datetime.now()

    # Mantém somente as colunas importantes
    df = df[
        [
            "ano",
            "id_municipio",
            "id_municipio_nome",
            "rede",
            "taxa_alfabetizacao",
            "ranking",
            "_gold_processed_at"
        ]
    ]

    log.info("Ranking de Municípios criado")

    return df

In [10]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - EVOLUÇÃO UF
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def gold_evolucao_uf():

    log.info("Construindo Gold: Evolução UF")

    df = ler_silver("uf")

    df = df[
        [
            "ano",
            "sigla_uf",
            "sigla_uf_nome",
            "rede",
            "taxa_alfabetizacao",
            "media_portugues"
        ]
    ].copy()

    df["_gold_processed_at"] = datetime.now()

    df = df.sort_values(
        by=["sigla_uf", "rede", "ano"]
    )

    log.info("Gold Evolução UF criada")

    return df

In [11]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - RESUMO POR REDE
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_resumo_rede():

    log.info("Construindo Gold: Resumo por Rede")

    df = ler_silver("uf")
    df = df[
        df["rede"] != "Total (Federal, Estadual, Municipal e Privada)"
    ]

    df_gold = (        
        df.groupby(["ano", "rede"])
          .agg(
              media_taxa_alfabetizacao=(
                  "taxa_alfabetizacao",
                  "mean"
              ),
              media_portugues=(
                  "media_portugues",
                  "mean"
              ),
              quantidade_ufs=(
                  "sigla_uf",
                  "nunique"
              )
          )
          .reset_index()
    )
    
    df_gold["media_taxa_alfabetizacao"] = (
    df_gold["media_taxa_alfabetizacao"].round(2)
    )

    df_gold["media_portugues"] = (
        df_gold["media_portugues"].round(2)
    )

    df_gold["_gold_processed_at"] = datetime.now()

    log.info("Resumo por Rede criado")

    return df_gold

In [12]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# HELPER: EXTRAI A META DO PRÓPRIO ANO DA LINHA
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# As tabelas de meta vêm em formato LARGO: uma coluna por ano-alvo
# (meta_alfabetizacao_2024 ... meta_alfabetizacao_2030) na mesma linha
# que carrega a taxa observada (`taxa_alfabetizacao`) daquele ano. Para
# comparar "resultado do ano X" com "meta do ano X", é preciso pegar,
# em cada linha, a coluna de meta cujo ano bate com o `ano` da própria
# linha.

def _extrair_meta_do_ano(df):
    """
    Args:
        df (pandas.DataFrame): Tabela de meta (uf ou município), contendo
            a coluna `ano` e as colunas `meta_alfabetizacao_2024..2030`.

    Returns:
        pandas.Series: Valor da meta definida para o próprio `ano` da
            linha (NaN se não houver meta definida para aquele ano, ex.:
            anos anteriores a 2024).
    """

    colunas_meta = {
        ano: f"meta_alfabetizacao_{ano}"
        for ano in range(2024, 2031)
        if f"meta_alfabetizacao_{ano}" in df.columns
    }

    meta_do_ano = pd.Series(np.nan, index=df.index, dtype="float64")

    for ano, coluna in colunas_meta.items():
        mascara = df["ano"] == ano
        meta_do_ano.loc[mascara] = df.loc[mascara, coluna]

    return meta_do_ano

In [13]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - COMPARAÇÃO META VS. RESULTADO (UF)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_comparacao_meta_uf():

    log.info("Construindo Gold: Comparação Meta vs. Resultado - UF")

    df = ler_silver("meta_alfabetizacao_uf")

    # removendo o Total, mesmo critério usado no ranking/evolução de UF
    df = df[
        df["rede"] != "Total (Federal, Estadual, Municipal e Privada)"
    ].copy()

    df["meta_do_ano"] = _extrair_meta_do_ano(df)

    # só faz sentido comparar quando existe meta definida para aquele ano
    df = df[df["meta_do_ano"].notna()].copy()

    df["diferenca_pp"] = (
        df["taxa_alfabetizacao"] - df["meta_do_ano"]
    ).round(2)

    df["atingiu_meta"] = df["diferenca_pp"] >= 0

    df["_gold_processed_at"] = datetime.now()

    df = df[
        [
            "ano",
            "sigla_uf",
            "sigla_uf_nome",
            "rede",
            "taxa_alfabetizacao",
            "meta_do_ano",
            "diferenca_pp",
            "atingiu_meta",
            "_gold_processed_at"
        ]
    ]

    df = df.sort_values(
        by=["sigla_uf", "rede", "ano"]
    )

    log.info("Gold Comparação Meta vs. Resultado (UF) criada")

    return df

In [14]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - COMPARAÇÃO META VS. RESULTADO (MUNICÍPIO)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def gold_comparacao_meta_municipio():

    log.info("Construindo Gold: Comparação Meta vs. Resultado - Município")

    df = ler_silver("meta_alfabetizacao_municipio")

    df = df[
        df["rede"] != "Total (Federal, Estadual, Municipal e Privada)"
    ].copy()

    df["meta_do_ano"] = _extrair_meta_do_ano(df)

    df = df[df["meta_do_ano"].notna()].copy()

    df["diferenca_pp"] = (
        df["taxa_alfabetizacao"] - df["meta_do_ano"]
    ).round(2)

    df["atingiu_meta"] = df["diferenca_pp"] >= 0

    df["_gold_processed_at"] = datetime.now()

    df = df[
        [
            "ano",
            "id_municipio",
            "id_municipio_nome",
            "rede",
            "taxa_alfabetizacao",
            "meta_do_ano",
            "diferenca_pp",
            "atingiu_meta",
            "nivel_alfabetizacao",
            "_gold_processed_at"
        ]
    ]

    df = df.sort_values(
        by=["id_municipio", "rede", "ano"]
    )

    log.info("Gold Comparação Meta vs. Resultado (Município) criada")

    return df

In [15]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# INTEGRIDADE REFERENCIAL DOS JOINS DE ENRIQUECIMENTO (Fase 3)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Mesma função usada na Silver para validar o join alunos+município+UF
# (ver checar_integridade_referencial no notebook Silver) - replicada
# aqui para os 4 novos joins de enriquecimento externo (Censo Escolar,
# Indicadores Educacionais, População, INSE), mantendo o mesmo padrão
# de rigor em vez de só um log informativo de cobertura.

def checar_integridade_referencial(df, coluna_fk, df_referencia, coluna_referencia, nome_relacao, critico=False):
    """
    Verifica se todo valor não nulo de `coluna_fk` em `df` existe em
    `coluna_referencia` de `df_referencia` (chave estrangeira).

    Args:
        df (pandas.DataFrame): Tabela "filha" (contém a chave estrangeira).
        coluna_fk (str): Coluna de `df` que deveria referenciar a outra tabela.
        df_referencia (pandas.DataFrame): Tabela "pai" (domínio válido).
        coluna_referencia (str): Coluna de `df_referencia` que serve de domínio.
        nome_relacao (str): Nome descritivo da relação, usado no log.
        critico (bool): Se True, levanta exceção quando houver órfãos.

    Raises:
        AssertionError: Se `critico=True` e existirem valores órfãos.
    """
    chaves_validas = set(df_referencia[coluna_referencia].dropna())
    valores = df[coluna_fk].dropna()

    orfaos = valores[~valores.isin(chaves_validas)]
    qtd_orfaos = len(orfaos)
    qtd_total = len(valores)
    percentual = round((qtd_orfaos / qtd_total) * 100, 2) if qtd_total else 0.0

    ok = qtd_orfaos == 0
    status = "PASS" if ok else ("FAIL" if critico else "WARN")
    detalhe = (
        f"{qtd_orfaos} de {qtd_total} ({percentual}%) valor(es) de '{coluna_fk}' "
        f"sem correspondência em '{nome_relacao}.{coluna_referencia}'"
    )

    if ok:
        log.info(f"[DQ:GOLD:REFERENCIAL] {status} | {detalhe}")
    elif critico:
        log.error(f"[DQ:GOLD:REFERENCIAL] {status} | {detalhe}")
        raise AssertionError(detalhe)
    else:
        log.warning(f"[DQ:GOLD:REFERENCIAL] {status} | {detalhe}")

    return {"ok": ok, "orfaos": qtd_orfaos, "total": qtd_total, "percentual": percentual}

In [16]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# AGREGAÇÃO DE INSE PARA NÍVEL DE MUNICÍPIO
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# DESCOBERTA (dado real): `id_escola` da tabela `alunos` NÃO é o código
# INEP oficial de escola (ex.: aluno tem id_escola=60000268, enquanto o
# INSE usa o código oficial, ex.: 31079332 - os 2 primeiros dígitos do
# código oficial são o código de UF, e "60" não é um código de UF
# válido; "31" é, de Minas Gerais). Ou seja, são sistemas de
# identificação diferentes - NÃO é possível fazer join por escola entre
# `alunos` e INSE.
#
# Solução adotada: agregar INSE para nível de MUNICÍPIO e juntar por
# (ano, id_municipio), perdendo granularidade de escola mas preservando
# o sinal socioeconômico. Essa perda de granularidade está documentada
# como limitação do projeto (ver README/modelo_dados.md).
#
# Censo Escolar foi SUBSTITUÍDO por PIB dos Municípios (br_ibge_pib) -
# que já nasce no nível de município, sem esse problema de chave.

def agregar_inse_por_municipio(df_inse):
    """
    Agrega o INSE (1 linha por escola) para 1 linha por
    (ano, id_municipio), usando a MÉDIA do inse. `classificacao` é
    descartada nesta agregação: é uma categorização derivada do próprio
    `inse` (faixas), redundante depois de agregado por média.

    Também calcula `quantidade_escolas_inse` (quantas escolas entraram
    na média) - um indicador de COBERTURA/confiabilidade do agregado:
    um município com 1 escola no INSE tem uma média bem menos robusta
    que um com 50. Não entra como feature no modelo inicialmente, mas
    fica disponível para quem quiser usar como peso ou filtro.

    Args:
        df_inse (pandas.DataFrame): Saída de ler_silver("inse_escola").

    Returns:
        pandas.DataFrame: 1 linha por (ano, id_municipio), com
            `inse_medio` e `quantidade_escolas_inse`.
    """

    df_agregado = (
        df_inse.groupby(["ano", "id_municipio"])["inse"]
        .agg(inse_medio="mean", quantidade_escolas_inse="count")
        .reset_index()
    )

    log.info(
        f"[AGREGACAO:INSE] {len(df_inse)} escola(s) agregada(s) em "
        f"{len(df_agregado)} combinação(ões) de (ano, id_municipio)"
    )

    return df_agregado

In [17]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GOLD - BASE DE ALUNOS PARA MODELAGEM (ALFABETIZAÇÃO)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Esta é a base de referência para a Fase 3 (modelo supervisionado que
# prevê `alfabetizado`). Vem da integração alunos + município + UF feita
# na Silver (`alunos_integrado`), no nível de granularidade certo: um
# registro por aluno, enriquecida com PIB dos Municípios, Indicadores
# Educacionais, População e INSE.

# AVISO DE DATA LEAKAGE: as colunas abaixo são mantidas nesta tabela por
# transparência/rastreabilidade, mas NÃO PODEM entrar como feature no
# treinamento do modelo -- cada uma vaza informação do próprio alvo.
COLUNAS_RISCO_DATA_LEAKAGE = {
    "proficiencia": (
        "É a variável usada para DEFINIR o alvo `alfabetizado` "
        "(corte de proficiência). Usá-la como feature vaza o alvo "
        "diretamente."
    ),
    "taxa_alfabetizacao_municipio": (
        "Agregado que já inclui o resultado do próprio aluno no cálculo "
        "(leakage indireto: o valor 'carrega' a resposta)."
    ),
    "taxa_alfabetizacao_uf": (
        "Mesmo motivo acima, em nível estadual."
    ),
    "media_portugues_municipio": (
        "Também é um agregado calculado a partir da proficiência dos "
        "próprios alunos do município (mesmo mecanismo de leakage "
        "indireto de taxa_alfabetizacao_municipio)."
    ),
    "media_portugues_uf": (
        "Mesmo motivo acima, em nível estadual."
    ),
}

# Indicadores de RESULTADO (aprovação/reprovação/abandono/distorção
# idade-série) do MESMO ano da avaliação seriam leakage direto -- são
# resultado do mesmo processo educacional que estamos tentando prever,
# só que agregado em nível municipal. Em vez de descartar essas
# variáveis (que carregam sinal real de contexto do município), usamos
# o valor do ANO ANTERIOR como "histórico recente", e já renomeamos a
# coluna com o sufixo `_ano_anterior` para deixar isso auto-documentado
# -- ninguém usa essa coluna sem saber que é defasada.
COLUNAS_INDICADORES_DEFASADAS = {
    "tdi_ef_2_ano": "tdi_ef_2_ano_anterior",
    "taxa_aprovacao_ef_2_ano": "taxa_aprovacao_ef_2_ano_anterior",
    "taxa_reprovacao_ef_2_ano": "taxa_reprovacao_ef_2_ano_anterior",
    "taxa_abandono_ef_2_ano": "taxa_abandono_ef_2_ano_anterior",
}


def gold_alunos_alfabetizacao():

    log.info("Construindo Gold: Base de Alunos para Modelagem (Alfabetização)")

    df = ler_silver("alunos_integrado")

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # Escopo do ano-base da modelagem (Fase 3)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # `alunos_integrado` (Silver) preserva todo o histórico disponível,
    # como as demais tabelas originais da Fase 2. A Gold é o lugar certo
    # para aplicar a regra de negócio de escopo: o modelo supervisionado
    # da Fase 3 é uma classificação TRANSVERSAL treinada e testada dentro
    # de um único ano-base (2023), não uma previsão temporal -- por isso
    # restringimos aqui, e não na extração (Bronze/Silver).
    ANO_BASE_MODELAGEM = 2023

    antes_ano = len(df)
    df = df[df["ano"] == ANO_BASE_MODELAGEM].copy()
    fora_do_escopo = antes_ano - len(df)

    if fora_do_escopo > 0:
        log.info(
            f"[ESCOPO] {fora_do_escopo} registro(s) fora do ano-base "
            f"{ANO_BASE_MODELAGEM} removido(s) (outros anos ficam "
            f"disponíveis em alunos_integrado, na Silver, caso "
            f"precisem ser usados depois)"
        )

    # mantém só alunos com o alvo definido: a avaliação não é aplicada a
    # 100% dos alunos (ausentes, caderno não preenchido etc.) -- sem
    # `alfabetizado` preenchido, o registro não serve para treino
    # supervisionado
    antes = len(df)
    df = df[df["alfabetizado"].notna()].copy()
    removidos = antes - len(df)

    if removidos > 0:
        log.info(
            f"Removidos {removidos} registro(s) sem alfabetizado definido "
            f"(aluno não avaliado)"
        )

    total_alunos = len(df)

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # 1) População (join direto por ano + id_municipio; sem risco de
    #    leakage). Feito ANTES do PIB de propósito: o PIB per capita
    #    (bloco 2) precisa da população já mesclada.
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    df_populacao = ler_silver("populacao_municipio")

    df = df.merge(df_populacao[["ano", "id_municipio", "populacao"]], on=["ano", "id_municipio"], how="left")

    checar_integridade_referencial(
        df, "id_municipio", df_populacao, "id_municipio",
        nome_relacao="populacao_municipio", critico=False
    )

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # 2) PIB dos Municípios (join direto por ano + id_municipio; nível
    #    nativo de município, sem precisar de agregação - diferente do
    #    Censo Escolar/INSE, que usam id_escola incompatível com `alunos`)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    df_pib = ler_silver("pib_municipio")

    # DESCOBERTA (dado real): impostos_liquidos, va, va_agropecuaria,
    # va_industria, va_servicos e va_adespss vieram 100% nulos para 2023
    # (confirmado nos 5.570 municípios) - o detalhamento setorial do PIB
    # tem defasagem maior que o total agregado, que já está disponível.
    # Mantemos só "pib" (usado também para calcular pib_per_capita), e
    # documentamos essa ausência como limitação do projeto.
    colunas_pib = [
        "ano", "id_municipio", "pib",
    ]
    colunas_pib = [c for c in colunas_pib if c in df_pib.columns]

    df = df.merge(df_pib[colunas_pib], on=["ano", "id_municipio"], how="left")

    checar_integridade_referencial(
        df, "id_municipio", df_pib, "id_municipio",
        nome_relacao="pib_municipio", critico=False
    )

    # PIB per capita: `pib` sozinho mistura tamanho do município com
    # riqueza (cidade grande tem PIB absoluto alto só por ser grande).
    # Dividir pela população (já mesclada no bloco 1) normaliza isso.
    if "pib" in df.columns and "populacao" in df.columns:
        df["pib_per_capita"] = df["pib"] / df["populacao"]

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # 2.5) Meta de Alfabetização Municipal (join por ano + id_municipio +
    #      rede). Adicionada após revisão do enunciado da Fase 3, que
    #      pede explicitamente "metas municipais" como pilar de dado e
    #      pergunta de negócio ("como prever municípios que podem não
    #      atingir metas futuras?").
    #
    #      IMPORTANTE: só trazemos `meta_alfabetizacao_2024` (meta do
    #      PRÓXIMO ano, decidida por política pública, independente do
    #      resultado do aluno de 2023). NÃO trazemos `taxa_alfabetizacao`
    #      dessa tabela - é o mesmo tipo de dado leakage que já excluímos
    #      em `taxa_alfabetizacao_municipio` (agregado que inclui o
    #      próprio aluno no cálculo), só vindo de outra tabela.
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    df_meta_municipio = ler_silver("meta_alfabetizacao_municipio")

    colunas_meta = ["ano", "id_municipio", "rede", "meta_alfabetizacao_2024"]
    colunas_meta = [c for c in colunas_meta if c in df_meta_municipio.columns]

    df = df.merge(df_meta_municipio[colunas_meta], on=["ano", "id_municipio", "rede"], how="left")
    df = df.rename(columns={"meta_alfabetizacao_2024": "meta_alfabetizacao_2024_municipio"})

    checar_integridade_referencial(
        df, "id_municipio", df_meta_municipio, "id_municipio",
        nome_relacao="meta_alfabetizacao_municipio", critico=False
    )

    sem_meta = df["meta_alfabetizacao_2024_municipio"].isna().sum()
    log.info(
        f"[ENRIQUECIMENTO:META_MUNICIPAL] {sem_meta} de {total_alunos} "
        f"({round(sem_meta / total_alunos * 100, 2)}%) aluno(s) sem meta "
        f"municipal correspondente (ano+id_municipio+rede)"
    )

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # 2.6) Meta de Alfabetização Estadual (join por ano + sigla_uf)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # DESCOBERTA (dado real): meta_alfabetizacao_uf usa rede="Pública"
    # (valor único, sem quebrar Municipal/Estadual) - diferente de
    # `alunos`, que usa Municipal/Estadual. Um join por rede exata
    # resultava em 100% de nulo. Como todos os nossos alunos já são de
    # rede pública (sem Federal/Privada na base), filtramos a fonte para
    # rede == "Pública" e juntamos só por (ano, sigla_uf).
    df_meta_uf = ler_silver("meta_alfabetizacao_uf")
    df_meta_uf = df_meta_uf[df_meta_uf["rede"] == "Pública"]

    colunas_meta_uf = ["ano", "sigla_uf", "meta_alfabetizacao_2024"]
    colunas_meta_uf = [c for c in colunas_meta_uf if c in df_meta_uf.columns]

    df = df.merge(df_meta_uf[colunas_meta_uf], on=["ano", "sigla_uf"], how="left")
    df = df.rename(columns={"meta_alfabetizacao_2024": "meta_alfabetizacao_2024_uf"})

    checar_integridade_referencial(
        df, "sigla_uf", df_meta_uf, "sigla_uf",
        nome_relacao="meta_alfabetizacao_uf", critico=False
    )

    sem_meta_uf = df["meta_alfabetizacao_2024_uf"].isna().sum()
    log.info(
        f"[ENRIQUECIMENTO:META_ESTADUAL] {sem_meta_uf} de {total_alunos} "
        f"({round(sem_meta_uf / total_alunos * 100, 2)}%) aluno(s) sem meta "
        f"estadual correspondente (ano+sigla_uf+rede)"
    )

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # 2.7) Meta de Alfabetização Nacional (join só por ano - mesmo motivo
    #      do bloco 2.6: rede="Pública" é valor único, sem quebra por
    #      Municipal/Estadual)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    df_meta_brasil = ler_silver("meta_alfabetizacao_brasil")
    df_meta_brasil = df_meta_brasil[df_meta_brasil["rede"] == "Pública"]

    colunas_meta_brasil = ["ano", "meta_alfabetizacao_2024"]
    colunas_meta_brasil = [c for c in colunas_meta_brasil if c in df_meta_brasil.columns]

    df = df.merge(df_meta_brasil[colunas_meta_brasil], on=["ano"], how="left")
    df = df.rename(columns={"meta_alfabetizacao_2024": "meta_alfabetizacao_2024_brasil"})

    sem_meta_brasil = df["meta_alfabetizacao_2024_brasil"].isna().sum()
    log.info(
        f"[ENRIQUECIMENTO:META_NACIONAL] {sem_meta_brasil} de {total_alunos} "
        f"({round(sem_meta_brasil / total_alunos * 100, 2)}%) aluno(s) sem meta "
        f"nacional correspondente (ano+rede)"
    )

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # 3) Indicadores Educacionais - parte ESTRUTURAL (mesmo ano; sem risco
    #    de leakage, pois refletem composição/formação docente e
    #    complexidade de gestão, não o resultado da avaliação)
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    df_indicadores = ler_silver("indicadores_educacionais_municipio")

    colunas_indicadores_estruturais = [
        "ano", "id_municipio",
        "atu_ef_anos_iniciais", "had_ef_anos_iniciais",
        "dsu_ef_anos_iniciais", "afd_ef_anos_iniciais_grupo_1",
        "ird_alta", "ird_baixa_regularidade",
        "icg_nivel_1", "icg_nivel_2", "icg_nivel_3",
        "icg_nivel_4", "icg_nivel_5", "icg_nivel_6",
    ]
    colunas_indicadores_estruturais = [c for c in colunas_indicadores_estruturais if c in df_indicadores.columns]

    df = df.merge(df_indicadores[colunas_indicadores_estruturais], on=["ano", "id_municipio"], how="left")

    checar_integridade_referencial(
        df, "id_municipio", df_indicadores, "id_municipio",
        nome_relacao="indicadores_educacionais_municipio (estrutural)", critico=False
    )

    # checar_integridade_referencial acima só valida que id_municipio É
    # UM CÓDIGO VÁLIDO na tabela de referência - não confirma que o join
    # composto (ano, id_municipio) realmente encontrou uma linha. Reforço
    # aqui com uma checagem direta de nulo numa coluna real do merge
    # (mesmo padrão já usado no bloco de indicadores defasados).
    if "atu_ef_anos_iniciais" in df.columns:
        sem_estrutural = df["atu_ef_anos_iniciais"].isna().sum()
        log.info(
            f"[ENRIQUECIMENTO:INDICADORES_EDUCACIONAIS] {sem_estrutural} de "
            f"{total_alunos} ({round(sem_estrutural / total_alunos * 100, 2)}%) "
            f"aluno(s) sem indicador estrutural no mesmo ano (confirma se o "
            f"join por ano+id_municipio realmente encontrou correspondência)"
        )

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # 4) Indicadores Educacionais - parte de RESULTADO HISTÓRICO (ano-1)
    #    Ver COLUNAS_INDICADORES_DEFASADAS acima para o motivo.
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    colunas_defasadas_disponiveis = [c for c in COLUNAS_INDICADORES_DEFASADAS if c in df_indicadores.columns]

    if colunas_defasadas_disponiveis:

        df_indicadores_defasados = df_indicadores[["ano", "id_municipio"] + colunas_defasadas_disponiveis].copy()

        # o indicador do ano X vira "contexto do ano anterior" para o ano X+1
        df_indicadores_defasados["ano"] = df_indicadores_defasados["ano"] + 1

        df_indicadores_defasados = df_indicadores_defasados.rename(columns=COLUNAS_INDICADORES_DEFASADAS)

        df = df.merge(df_indicadores_defasados, on=["ano", "id_municipio"], how="left")

        primeira_coluna_defasada = COLUNAS_INDICADORES_DEFASADAS[colunas_defasadas_disponiveis[0]]
        sem_defasado = df[primeira_coluna_defasada].isna().sum()

        log.info(
            f"[ENRIQUECIMENTO:INDICADORES_EDUCACIONAIS] {sem_defasado} de "
            f"{total_alunos} ({round(sem_defasado / total_alunos * 100, 2)}%) "
            f"aluno(s) sem indicador do ano anterior (ex.: primeiro ano "
            f"disponível na série histórica, sem ano-1 para comparar)"
        )

    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    # 5) INSE - Nível Socioeconômico, agregado por MUNICÍPIO
    #    id_escola de `alunos` não é o código INEP oficial (ver nota no
    #    topo do notebook) - agregado por (ano, id_municipio) via
    #    agregar_inse_por_municipio. Não é leakage (reflete a família,
    #    não o resultado de alfabetização) e não precisa de defasagem.
    # ~~~~~~~~~~~~~~~~~~~~~~~~~~~~
    df_inse = ler_silver("inse_escola")
    df_inse_municipio = agregar_inse_por_municipio(df_inse)

    df = df.merge(df_inse_municipio, on=["ano", "id_municipio"], how="left")

    checar_integridade_referencial(
        df, "id_municipio", df_inse_municipio, "id_municipio",
        nome_relacao="inse_escola (agregado por município)", critico=False
    )

    df["_gold_processed_at"] = datetime.now()

    colunas_finais = [
        "ano",
        "id_aluno",
        "id_escola",
        "id_municipio",
        "id_municipio_nome",
        "sigla_uf",
        "sigla_uf_nome",
        "serie",
        "rede",
        "presenca",
        "preenchimento_caderno",
        "peso_aluno",
        "taxa_alfabetizacao_municipio",
        "media_portugues_municipio",
        "taxa_alfabetizacao_uf",
        "media_portugues_uf",
        "proficiencia",
        # População
        "populacao",
        # PIB dos Municípios (só o total - detalhamento setorial não
        # disponível para 2023, ver nota acima)
        "pib",
        "pib_per_capita",
        # Metas de Alfabetização (próximo ano) - municipal, estadual, nacional
        "meta_alfabetizacao_2024_municipio",
        "meta_alfabetizacao_2024_uf",
        "meta_alfabetizacao_2024_brasil",
        # Indicadores Educacionais (estruturais)
        "atu_ef_anos_iniciais",
        "had_ef_anos_iniciais",
        "dsu_ef_anos_iniciais",
        "afd_ef_anos_iniciais_grupo_1",
        "ird_alta",
        "ird_baixa_regularidade",
        "icg_nivel_1",
        "icg_nivel_2",
        "icg_nivel_3",
        "icg_nivel_4",
        "icg_nivel_5",
        "icg_nivel_6",
        # Indicadores Educacionais (defasados, ano anterior)
        "tdi_ef_2_ano_anterior",
        "taxa_aprovacao_ef_2_ano_anterior",
        "taxa_reprovacao_ef_2_ano_anterior",
        "taxa_abandono_ef_2_ano_anterior",
        # INSE (Nível Socioeconômico, agregado por município)
        "inse_medio",
        "quantidade_escolas_inse",
        # Alvo
        "alfabetizado",
        "_gold_processed_at",
    ]

    colunas_finais = [c for c in colunas_finais if c in df.columns]

    df = df[colunas_finais]

    log.warning(
        f"[DATA LEAKAGE] Esta tabela contém colunas que NÃO podem ser "
        f"usadas como feature: {list(COLUNAS_RISCO_DATA_LEAKAGE.keys())}. "
        f"Ver COLUNAS_RISCO_DATA_LEAKAGE para o motivo de cada uma."
    )

    log.info(
        f"Gold Base de Alunos para Modelagem criada: {len(df)} registro(s)"
    )

    return df

In [18]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# REGRAS DE DATA QUALITY
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
CHECKS = {

    "gold_ranking_uf": [

        {
            "tipo": "min_count",
            "valor": 100,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "sigla_uf",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "ranking",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0,100),
            "critico": True
        }

    ],
    "ranking_municipio": [

        {
            "tipo": "min_count",
            "valor": 1000,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "id_municipio",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "ranking",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0, 100),
            "critico": True
        }
    ],
    "evolucao_uf": [

        {
            "tipo": "min_count",
            "valor": 100,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "sigla_uf",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "ano",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0, 100),
            "critico": True
        }

    ],
    "resumo_rede": [

    {
        # Ajustado de 7 para 4 após execução com dado real: o valor 7 era
        # uma suposição da Fase 2 que não se confirmou -- na prática, a
        # base do INEP cobre 1 único ano de avaliação, e resumo_rede
        # agrupa por (ano, rede), então o volume real reflete apenas a
        # quantidade de redes de ensino distintas (~6), não 7. Mantém 4
        # como piso de sanidade (garante que a agregação não veio vazia
        # ou drasticamente incompleta), tolerando crescimento futuro se
        # a fonte passar a cobrir mais anos.
        "tipo": "min_count",
        "valor": 4,
        "critico": True
    },

    {
        "tipo": "not_null",
        "coluna": "ano",
        "critico": True
    },

    {
        "tipo": "not_null",
        "coluna": "rede",
        "critico": True
    },

    {
        "tipo": "range",
        "coluna": "media_taxa_alfabetizacao",
        "valor": (0, 100),
        "critico": True
    }

],

    "comparacao_meta_uf": [

        {
            # valor conservador: como só existem metas para 2024-2030,
            # o volume real depende de quantos desses anos já têm
            # avaliação (ano) registrada na base. Ajustar após a
            # primeira execução com dados reais.
            "tipo": "min_count",
            "valor": 10,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "sigla_uf",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "meta_do_ano",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0, 100),
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "meta_do_ano",
            "valor": (0, 100),
            "critico": True
        }

    ],

    "comparacao_meta_municipio": [

        {
            "tipo": "min_count",
            "valor": 50,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "id_municipio",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "meta_do_ano",
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "taxa_alfabetizacao",
            "valor": (0, 100),
            "critico": True
        },

        {
            "tipo": "range",
            "coluna": "meta_do_ano",
            "valor": (0, 100),
            "critico": True
        }

    ],

    "alunos_alfabetizacao": [

        {
            "tipo": "min_count",
            "valor": 100,
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "id_aluno",
            "critico": True
        },

        {
            "tipo": "not_null",
            "coluna": "alfabetizado",
            "critico": True
        },

        {
            "tipo": "unique",
            "coluna": "id_aluno",
            "critico": False
        }

    ]

}

In [19]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNÇÃO DE QUALIDADE DA CAMADA GOLD
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def checar_qualidade(df, checks):

    log.info("Iniciando verificações de qualidade")

    for check in checks:

        if check["tipo"] == "min_count":

            assert len(df) >= check["valor"], \
                f"Quantidade mínima não atendida ({len(df)} registros)."

        elif check["tipo"] == "not_null":

            coluna = check["coluna"]

            assert df[coluna].isnull().sum() == 0, \
                f"Existem valores nulos na coluna '{coluna}'."

        elif check["tipo"] == "range":

            coluna = check["coluna"]
            minimo, maximo = check["valor"]

            assert (
                df[coluna].between(minimo, maximo).all()
            ), f"Valores fora do intervalo na coluna '{coluna}'."

    log.info("Checks de qualidade concluídos com sucesso!")

In [20]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# SALVA UM DATAFRAME NA CAMADA GOLD EM FORMATO PARQUET
"""
    Args:
        df (pandas.DataFrame): DataFrame tratado.
"""
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def salvar_gold(df, nome):

    caminho = DATA_GOLD / f"{nome}.parquet"

    df.to_parquet(
        caminho,
        index=False
    )

    log.info(f"Camada GOLD salva em {caminho}")

    return caminho

In [21]:
# ~~~~~~~~~~~~~~~~~~~~~~~~
# EXECUÇÃO DA CAMADA GOLD
"""
    Cada dataset Gold é construído de forma isolada: se um deles falhar,
    um alerta é emitido e os demais continuam sendo processados.
    Latência e volume de cada dataset são registrados como métricas
    estruturadas e consultáveis.
"""
# ~~~~~~~~~~~~~~~~~~~~~~~~
def executar_gold():

    log.info("~" * 35)
    log.info("INICIANDO CAMADA GOLD")
    log.info("~" * 35)

    inicio_pipeline = time.perf_counter()

    # Cada item: (nome do dataset salvo, função que constrói o dataframe, chave em CHECKS)
    datasets = [
        ("ranking_uf", gold_ranking_uf, "gold_ranking_uf"),
        ("ranking_municipio", gold_ranking_municipio, "ranking_municipio"),
        ("evolucao_uf", gold_evolucao_uf, "evolucao_uf"),
        ("resumo_rede", gold_resumo_rede, "resumo_rede"),
        ("comparacao_meta_uf", gold_comparacao_meta_uf, "comparacao_meta_uf"),
        ("comparacao_meta_municipio", gold_comparacao_meta_municipio, "comparacao_meta_municipio"),
        ("alunos_alfabetizacao", gold_alunos_alfabetizacao, "alunos_alfabetizacao"),
    ]

    datasets_ok = 0
    datasets_falha = 0

    for nome, funcao_construtora, chave_checks in datasets:

        log.info(f"Checando qualidade de {nome}")

        inicio_dataset = time.perf_counter()

        try:

            df_gold = funcao_construtora()

            checar_qualidade(
                df_gold,
                CHECKS[chave_checks]
            )

            salvar_gold(
                df_gold,
                nome
            )

            latencia_segundos = round(time.perf_counter() - inicio_dataset, 2)

            log_metrica(
                "tabela_processada",
                camada="gold",
                tabela=nome,
                volume=len(df_gold),
                latencia_segundos=latencia_segundos,
                status="sucesso"
            )

            datasets_ok += 1

        except Exception as e:

            latencia_segundos = round(time.perf_counter() - inicio_dataset, 2)

            log_metrica(
                "tabela_processada",
                camada="gold",
                tabela=nome,
                volume=0,
                latencia_segundos=latencia_segundos,
                status="falha"
            )

            emitir_alerta(
                f"Falha na construção do dataset Gold '{nome}'",
                camada="gold",
                tabela=nome,
                erro=str(e)
            )

            datasets_falha += 1

            # isola a falha: segue para o próximo dataset
            continue

    latencia_total_segundos = round(time.perf_counter() - inicio_pipeline, 2)

    log_metrica(
        "pipeline_concluido",
        camada="gold",
        tabelas_ok=datasets_ok,
        tabelas_falha=datasets_falha,
        latencia_total_segundos=latencia_total_segundos
    )

    if datasets_falha > 0:
        emitir_alerta(
            f"Pipeline Gold concluído com {datasets_falha} falha(s) de {datasets_ok + datasets_falha} dataset(s)",
            camada="gold",
            datasets_falha=datasets_falha,
            datasets_ok=datasets_ok
        )

    log.info("Camada Gold concluída!")

In [22]:
executar_gold()

2026-09-06 21:54:58,375 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-09-06 21:54:58,376 | INFO     | INICIANDO CAMADA GOLD
2026-09-06 21:54:58,377 | INFO     | ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
2026-09-06 21:54:58,378 | INFO     | Checando qualidade de ranking_uf
2026-09-06 21:54:58,379 | INFO     | Construindo Gold: Ranking UF
2026-09-06 21:54:58,380 | INFO     | Lendo Silver: silver\uf.parquet
2026-09-06 21:54:58,430 | INFO     | Ranking UF criado
2026-09-06 21:54:58,431 | INFO     | Iniciando verificações de qualidade
2026-09-06 21:54:58,432 | INFO     | Checks de qualidade concluídos com sucesso!
2026-09-06 21:54:58,436 | INFO     | Camada GOLD salva em gold\ranking_uf.parquet
2026-09-06 21:54:58,437 | INFO     | [METRICA] evento=tabela_processada | camada=gold | tabela=ranking_uf | volume=144 | latencia_segundos=0.06 | status=sucesso
2026-09-06 21:54:58,437 | INFO     | Checando qualidade de ranking_municipio
2026-09-06 21:54:58,438 | INFO     | Construindo Gold: Rankin